In [ ]:
Import Libraries

In [25]:
# Importing libraries

import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, f1_score, confusion_matrix,
                             ConfusionMatrixDisplay, precision_recall_curve)


warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

try:
    OHE = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    OHE = OneHotEncoder(handle_unknown="ignore", sparse=True)


Load training Data

In [28]:
# Load the training data
df_raw = pd.read_csv(
    "/Users/cyrildaisychristina/AIGC/5005 Capstone Project Prep/Midterm/risk-train.txt",
    sep="\t",
    dtype=str,
    na_values=["?", "NA", "NaN", ""],
    keep_default_na=False
)

df_raw.shape
df_raw.head()


,ORDER_ID,CLASS,B_EMAIL,B_TELEFON,B_BIRTHDATE,FLAG_LRIDENTISCH,FLAG_NEWSLETTER,Z_METHODE,Z_CARD_ART,Z_CARD_VALID,...,FAIL_RPLZ,FAIL_RORT,FAIL_RPLZORTMATCH,SESSION_TIME,NEUKUNDE,AMOUNT_ORDER_PRE,VALUE_ORDER_PRE,DATE_LORDER,MAHN_AKT,MAHN_HOECHST
0,49917,no,yes,no,1/17/1973,yes,yes,check,NaN,5.2006,...,no,no,no,8,yes,0,0,NaN,NaN,NaN
1,49919,no,yes,yes,12/8/1970,no,no,credit_card,Visa,12.2007,...,yes,no,no,13,yes,0,0,NaN,NaN,NaN
2,49923,no,yes,no,4/3/1972,yes,no,check,NaN,12.2007,...,no,no,no,3,yes,0,0,NaN,NaN,NaN
3,49924,no,no,yes,8/1/1966,yes,no,check,NaN,1.2007,...,no,no,no,11,no,4,75.72,5/12/2002,0,0
4,49927,no,yes,yes,12/21/1969,yes,no,credit_card,Eurocard,12.2006,...,no,no,no,16,yes,0,0,NaN,NaN,NaN


In [29]:
print(df_raw["CLASS"].value_counts(dropna=False))

CLASS
no     28254
yes     1746
Name: count, dtype: int64


In [30]:
df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]

# Map yes/no columns 
binary_cols = []
for c in df.columns:
    if c == "CLASS":
        continue
    uniq = set(str(x).lower() for x in df[c].dropna().unique())
    if len(uniq) > 0 and uniq.issubset({"yes", "no"}):
        binary_cols.append(c)

yn_map = {"yes": 1, "no": 0}
for c in binary_cols:
    df[c] = df[c].str.lower().map(yn_map).astype("float")

# Numeric columns 
numeric_like = [
    "VALUE_ORDER", "AMOUNT_ORDER", "SESSION_TIME",
    "AMOUNT_ORDER_PRE", "VALUE_ORDER_PRE", "MAHN_AKT", "MAHN_HOECHST"
]
for c in numeric_like:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# TIME_ORDER 
def parse_time_to_features(s):
    if pd.isna(s): return np.nan, np.nan
    parts = str(s).split(":")
    if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
        return int(parts[0]), int(parts[1])
    return np.nan, np.nan

if "TIME_ORDER" in df.columns:
    hours, minutes = zip(*df["TIME_ORDER"].apply(parse_time_to_features))
    df["HOUR"], df["MINUTE"] = hours, minutes

# WEEKDAY_ORDER
if "WEEKDAY_ORDER" in df.columns:
    df["WEEKDAY_ORDER"] = df["WEEKDAY_ORDER"].str.strip().str.capitalize()

# Z_CARD_VALID 
def parse_card_valid(x):
    m = re.match(r"^(\d{1,2})\.(\d{4})$", str(x))
    return (int(m.group(1)), int(m.group(2))) if m else (np.nan, np.nan)

if "Z_CARD_VALID" in df.columns:
    mths, yrs = zip(*df["Z_CARD_VALID"].apply(parse_card_valid))
    df["CARD_VALID_MONTH"], df["CARD_VALID_YEAR"] = mths, yrs

# B_BIRTHDATE -> approximate age at order (use card valid year or 2006 as coarse proxy)
if "B_BIRTHDATE" in df.columns:
    birth_year = pd.to_datetime(df["B_BIRTHDATE"], errors="coerce").dt.year
    ref_year = pd.Series(df.get("CARD_VALID_YEAR", pd.Series(index=df.index))).fillna(2006)
    df["BIRTH_YEAR"] = birth_year
    df["APPROX_AGE"] = ref_year - birth_year

# ANUMMER/ANUMBER item code coverage 
item_cols = [c for c in df.columns if c.startswith(("ANUMMER_", "ANUMMER", "ANUMBER_"))]
if item_cols:
    df["N_ITEMS_WITH_CODES"] = df[item_cols].notna().sum(axis=1)

# Separate features/target
X = df.drop(columns=["CLASS", "ORDER_ID"], errors="ignore")
y = df_raw["CLASS"].str.lower().map({"yes": 1, "no": 0}).astype(int)

print(f"Binary yes/no columns mapped: {len(binary_cols)}")
print("Feature sample:", X.columns[:15].tolist(), "...")

Binary yes/no columns mapped: 18
Feature sample: ['B_EMAIL', 'B_TELEFON', 'B_BIRTHDATE', 'FLAG_LRIDENTISCH', 'FLAG_NEWSLETTER', 'Z_METHODE', 'Z_CARD_ART', 'Z_CARD_VALID', 'Z_LAST_NAME', 'VALUE_ORDER', 'WEEKDAY_ORDER', 'TIME_ORDER', 'AMOUNT_ORDER', 'ANUMMER_01', 'ANUMMER_02'] ...


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print("Train size:", X_train.shape, " Test size:", X_test.shape)
print("Train class distribution:\n", y_train.value_counts(normalize=True))
print("Test  class distribution:\n", y_test.value_counts(normalize=True))

Train size: (24000, 49)  Test size: (6000, 49)
Train class distribution:
 CLASS
0    0.941792
1    0.058208
Name: proportion, dtype: float64
Test  class distribution:
 CLASS
0    0.941833
1    0.058167
Name: proportion, dtype: float64


In [31]:
# Simple random oversampling 
tr = X_train.copy()
tr["target"] = y_train
pos = tr[tr["target"] == 1]
neg = tr[tr["target"] == 0]

pos_over = resample(pos, replace=True, n_samples=len(neg), random_state=RANDOM_STATE)
train_balanced = pd.concat([neg, pos_over]).sample(frac=1.0, random_state=RANDOM_STATE)

X_train_bal = train_balanced.drop(columns=["target"])
y_train_bal = train_balanced["target"]
print("Balanced train size:", X_train_bal.shape, y_train_bal.mean())

Balanced train size: (45206, 49) 0.5


In [32]:
# Split columns by type
categorical_cols, numeric_cols = [], []
for c in X.columns:
    if c in binary_cols:
        numeric_cols.append(c)                # already 0/1 numeric
    elif c in ["HOUR","MINUTE","CARD_VALID_MONTH","CARD_VALID_YEAR",
               "BIRTH_YEAR","APPROX_AGE","VALUE_ORDER","AMOUNT_ORDER",
               "SESSION_TIME","AMOUNT_ORDER_PRE","VALUE_ORDER_PRE",
               "MAHN_AKT","MAHN_HOECHST","N_ITEMS_WITH_CODES"]:
        numeric_cols.append(c)
    elif X[c].dtype.kind in "biufc":
        numeric_cols.append(c)
    else:
        categorical_cols.append(c)

# Manage high-cardinality categoricals using simple frequency encoding 
cardinality = {c: X[c].nunique(dropna=True) for c in categorical_cols}
low_card_cats  = [c for c in categorical_cols if cardinality[c] <= 20]
high_card_cats = [c for c in categorical_cols if cardinality[c] >  20]

def fit_freq_maps(X_df):
    return {c: X_df[c].value_counts(normalize=True) for c in X_df.columns}

def freq_encode_apply(X_df, maps):
    out = pd.DataFrame(index=X_df.index)
    for c in X_df.columns:
        out[c + "_freq"] = X_df[c].map(maps[c]).fillna(0.0)
    return out

# Fit frequency maps on the (unbalanced) training portion only to prevent leakage
freq_maps = fit_freq_maps(X_train[high_card_cats]) if high_card_cats else {}

numeric_transformer = SimpleImputer(strategy="median")
onehot_transformer  = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("ohe", OHE)
])
freq_transformer    = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("freq", FunctionTransformer(lambda X_: freq_encode_apply(
        pd.DataFrame(X_, columns=high_card_cats), freq_maps
    ), validate=False))
]) if high_card_cats else "drop"

preprocessor = ColumnTransformer(
    transformers=[
        ("num",   numeric_transformer, numeric_cols),
        ("onehot", onehot_transformer, low_card_cats),
        ("freq",  freq_transformer,    high_card_cats)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("Numeric:", len(numeric_cols), " Low-card cats:", len(low_card_cats), " High-card cats:", len(high_card_cats))


Numeric: 32  Low-card cats: 5  High-card cats: 12


In [19]:
models = {
    "logreg": LogisticRegression(max_iter=2000, solver="lbfgs"),
    "rf":     RandomForestClassifier(n_estimators=300, n_jobs=-1,
                                     class_weight="balanced_subsample", random_state=RANDOM_STATE),
    "gb":     GradientBoostingClassifier(random_state=RANDOM_STATE),
}

results = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("clf", model)])
    pipe.fit(X_train_bal, y_train_bal)
    y_pred = pipe.predict(X_test)
    f1_yes = f1_score(y_test, y_pred, pos_label=1)
    results[name] = {"f1_yes": f1_yes, "pipe": pipe,
                     "report": classification_report(y_test, y_pred, target_names=["no","yes"])}
    print(f"{name}: F1(yes) = {f1_yes:.3f}")

best_model_name = max(results, key=lambda k: results[k]["f1_yes"])
best_model = results[best_model_name]["pipe"]
print("\nBest model by F1(yes):", best_model_name)
print(results[best_model_name]["report"])

logreg: F1(yes) = 0.182
rf: F1(yes) = 0.000
gb: F1(yes) = 0.224

Best model by F1(yes): gb
              precision    recall  f1-score   support

          no       0.97      0.75      0.85      5651
         yes       0.14      0.64      0.22       349

    accuracy                           0.74      6000
   macro avg       0.55      0.69      0.53      6000
weighted avg       0.92      0.74      0.81      6000



In [37]:
# 60/20/20 split: use a validation slice from the original training for threshold tuning
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=RANDOM_STATE
)

# Oversample only on the inner training split
tr2 = X_tr.copy(); tr2["target"] = y_tr
pos2, neg2 = tr2[tr2["target"]==1], tr2[tr2["target"]==0]
pos_over2  = resample(pos2, replace=True, n_samples=len(neg2), random_state=RANDOM_STATE)
tr2_bal    = pd.concat([neg2, pos_over2]).sample(frac=1.0, random_state=RANDOM_STATE)
X_tr2_bal  = tr2_bal.drop(columns=["target"])
y_tr2_bal  = tr2_bal["target"]

# Rebuild the preprocessor with freq maps fit on X_tr only
freq_maps_val = fit_freq_maps(X_tr[high_card_cats]) if high_card_cats else {}
preprocessor_val = ColumnTransformer(
    transformers=[
        ("num",   SimpleImputer(strategy="median"), numeric_cols),
        ("onehot", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OHE)]), low_card_cats),
        ("freq",  Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                            ("freq", FunctionTransformer(lambda X_: freq_encode_apply(
                                pd.DataFrame(X_, columns=high_card_cats), freq_maps_val
                            ), validate=False))]), high_card_cats)
    ],
    remainder="drop", verbose_feature_names_out=False
)

# Use the best estimator family from the quick sweep above (GB)
gb_val = Pipeline([("prep", preprocessor_val), ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))])
gb_val.fit(X_tr2_bal, y_tr2_bal)

# Threshold tuning for F1(yes) on validation
val_proba = gb_val.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0.05, 0.95, 19)
f1s = [f1_score(y_val, (val_proba >= t).astype(int), pos_label=1) for t in thresholds]

best_idx = int(np.argmax(f1s))
best_thr = float(thresholds[best_idx])
print(f"Best validation threshold = {best_thr:.2f}  --> F1(yes) on val = {f1s[best_idx]:.3f}")

# Retrain on the full outer training set (with oversampling), evaluate on test at tuned threshold
freq_maps_full = fit_freq_maps(X_train[high_card_cats]) if high_card_cats else {}
preprocessor_full = ColumnTransformer(
    transformers=[
        ("num",   SimpleImputer(strategy="median"), numeric_cols),
        ("onehot", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OHE)]), low_card_cats),
        ("freq",  Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                            ("freq", FunctionTransformer(lambda X_: freq_encode_apply(
                                pd.DataFrame(X_, columns=high_card_cats), freq_maps_full
                            ), validate=False))]), high_card_cats)
    ],
    remainder="drop", verbose_feature_names_out=False
)

gb_full = Pipeline([("prep", preprocessor_full), ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))])
gb_full.fit(X_train_bal, y_train_bal)

test_proba = gb_full.predict_proba(X_test)[:, 1]
y_test_pred_tuned = (test_proba >= best_thr).astype(int)
print(classification_report(y_test, y_test_pred_tuned, target_names=["no","yes"]))

# Save artifacts for the report
os.makedirs("artifacts", exist_ok=True)
# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred_tuned)
fig, ax = plt.subplots(figsize=(4,4))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["no","yes"]).plot(ax=ax, colorbar=False)
ax.set_title(f"Confusion Matrix (GB @ thr={best_thr:.2f})")
plt.savefig("artifacts/confusion_matrix.png", bbox_inches="tight")
plt.close(fig)

# PR curve on validation to visualize the trade-off we optimized
from sklearn.metrics import precision_recall_curve
prec, rec, thr = precision_recall_curve(y_val, val_proba)
plt.figure(figsize=(5,4))
plt.plot(rec, prec)
plt.xlabel("Recall (validation)")
plt.ylabel("Precision (validation)")
plt.title("Precision–Recall curve (validation)")
plt.savefig("artifacts/pr_curve_validation.png", bbox_inches="tight")
plt.close()

# Small JSON summary for reproducibility
summary = {
    "shape": df_raw.shape,
    "class_counts": df_raw["CLASS"].value_counts(dropna=False).to_dict(),
    "best_threshold": best_thr,
    "test_f1_yes_at_tuned_thr": f1_score(y_test, y_test_pred_tuned, pos_label=1),
}
with open("artifacts/model_results.json", "w") as f:
    json.dump(summary, f, indent=2)


Best validation threshold = 0.60  --> F1(yes) on val = 0.242
              precision    recall  f1-score   support

          no       0.96      0.85      0.90      5651
         yes       0.16      0.49      0.25       349

    accuracy                           0.83      6000
   macro avg       0.56      0.67      0.57      6000
weighted avg       0.92      0.83      0.86      6000

